## Análise exploratória dos dados - Jogos da Copa do Mundo de 2014 e 2026

Esta análise exploratória tem como objetivo responder perguntas sobre a geração de energia nos dias de Copa do Mundo, dos anos 2014 e 2026.

## 1. Introdução

Nesta análise, pretendemos responder algumas perguntas potencialmente relevantes, e validar algumas hipóteses, como:

- Na média, qual subsistema de energia foi mais afetado nos períodos de jogos da Copa do Mundo?
- A geração média é maior nesses períodos de Copa do Mundo?
- Hipótese: nestes períodos, é válido pensar que a geração térmica tem um pico de utilização (para suprir as demandas do sistema)?

## 2. Obtenção dos dados, limpeza e tratamento de dados

- **Fonte dos dados**: os dados foram obtidos a partir dos arquivos ``.csv`` encontrados em https://dados.ons.org.br/dataset/balanco-energia-subsistema, e então, carregados conforme o script ``load_datasets.py`` abaixo:



In [ ]:
# load_datasets.py
# Importações

import pandas as pd
import os
import glob

# Usar glob para conseguir todos os .csvs na pasta do repo
path = "C:\\Users\\Gustavo\\Desktop\\case_echoenergia\\datasets"
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Inserir todos os dfs em uma lista
df = [pd.read_csv(f, sep=';') for f in csv_files]

# Concatenar todos os dataframes em um consolidado
df_consolidado = pd.concat(df)

# Transformar df em .parquet
df_consolidado.to_parquet('df_consolidado.parquet', engine='pyarrow')

df_parquet = pd.read_parquet('df_consolidado.parquet')

Então, geramos a base consolidada em ``.parquet``

In [25]:
df_parquet.head()

,id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio,ano,mes
0,NE,NORDESTE,2000-01-01,5337.7,0.0,0.0,0.0,5340.2,-2.5,2000,1
1,N,NORTE,2000-01-01,2422.5,0.0,0.0,0.0,2373.7,48.8,2000,1
2,SIN,SISTEMA INTERLIGADO NACIONAL,2000-01-01,31977.0,2649.1,0.0,0.0,34673.9,-47.8,2000,1
3,SE,SUDESTE/CENTRO-OESTE,2000-01-01,20479.8,1418.6,0.0,0.0,21183.0,715.4,2000,1
4,S,SUL,2000-01-01,3737.0,1230.5,0.0,0.0,5777.0,-809.5,2000,1


### Dicionário de dados

O dicionário de dados pode ser encontrado em: ``DicionarioDados_Balanco_Energia_Subsistema.pdf``

### Limpeza e tratamento de dados

A base de dados já é fornecida relativamente bem tratada e limpa, com exceção de alguns valores nulos nas colunas ``val_gereolica`` e ``val_gersolar`` em anos mais antigos, então executamos:

In [ ]:
df_parquet = df_parquet.fillna(0)

df_parquet.head()

## 3. Análise Exploratória dos Dados

Agora partimos para uma análise mais profunda dos dados, em relação aos anos de 2014, temos: 

In [ ]:
# Importações básicas

import numpy as np
import pandas as pd 

In [86]:
# Criação de colunas de ano e mes

df_parquet['din_instante'] = pd.to_datetime(df_parquet['din_instante'])
df_parquet['ano'], df_parquet['mes'] = df_parquet['din_instante'].dt.year, df_parquet['din_instante'].dt.month

**(i)**

- Qual subsistema de energia foi mais afetado nos períodos de jogos da Copa do Mundo?

In [38]:
df_parquet

,id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio,ano,mes
0,NE,NORDESTE,2000-01-01 00:00:00,5337.700,0.000,0.000,0.0,5340.200,-2.500,2000,1
1,N,NORTE,2000-01-01 00:00:00,2422.500,0.000,0.000,0.0,2373.700,48.800,2000,1
2,SIN,SISTEMA INTERLIGADO NACIONAL,2000-01-01 00:00:00,31977.000,2649.100,0.000,0.0,34673.900,-47.800,2000,1
3,SE,SUDESTE/CENTRO-OESTE,2000-01-01 00:00:00,20479.800,1418.600,0.000,0.0,21183.000,715.400,2000,1
4,S,SUL,2000-01-01 00:00:00,3737.000,1230.500,0.000,0.0,5777.000,-809.500,2000,1
...,...,...,...,...,...,...,...,...,...,...,...
23035,N,NORTE,2026-07-11 23:00:00,3265.073,1995.982,338.086,0.0,9000.523,-3401.382,2026,7
23036,NE,NORDESTE,2026-07-11 23:00:00,4095.415,275.357,17049.217,0.0,13805.934,7614.055,2026,7
23037,S,SUL,2026-07-11 23:00:00,11487.781,1231.310,686.080,0.0,11750.251,1654.920,2026,7
23038,SE,SUDESTE/CENTRO-OESTE,2026-07-11 23:00:00,25296.739,6847.953,49.958,0.0,38062.243,-5867.593,2026,7


In [78]:
# Segmentação do DataFrame nos anos de Copa

df_parquet_2014 = df_parquet[df_parquet['ano'] == 2014]
df_parquet_2026 = df_parquet[df_parquet['ano'] == 2026]

In [64]:
# Agrupamento por subsistema

df_parquet_2014.groupby(['id_subsistema'])['val_carga'].mean()

id_subsistema
N       5136.825208
NE      9544.662238
S      10569.272473
SE     36142.702450
SIN    61386.396465
Name: val_carga, dtype: float64

In [85]:
# Agrupamento por subsistema
# Período de jogos do Brasil

df_parquet_2026['din_instante'] = pd.to_datetime(df_parquet_2026['din_instante'])
df_filtrado = df_parquet_2026.loc[(df_parquet_2026['mes'] >= 6) 
                                  & (df_parquet_2026['mes'] <= 7)]
df_filtrado.groupby('id_subsistema')['val_carga'].mean()

id_subsistema
N       8534.386982
NE     12974.957004
S      13355.577886
SE     40558.697704
SIN    75423.619990
Name: val_carga, dtype: float64

Observamos que, em média:

- No ano de 2014, os subsistemas mais afetados foram o SE (correspondente à região Sudeste) e o S (correspondente à região Sul)
- No ano de 2026, temos um comportamento parecido, no entanto, o subsistema NE (correspondente à região Nordeste) tem um uso de energia muito próximo ao do subsistema SUL

**(ii)**

- A geração média é maior nesses períodos de Copa do Mundo?

In [ ]:
df_parquet.groupby(['ano'])['val_carga'].mean()

ano
2000    16324.953494
2001    15091.838962
2002    15867.120650
2003    16705.805613
2004    17492.682439
2005    18285.428782
2006    18988.487396
2007    19892.244828
2008    20403.243903
2009    20239.231354
2010    21624.707363
2011    22411.941648
2012    23240.056629
2013    23433.349175
2014    24565.777114
2015    24485.449668
2016    24618.454901
2017    25072.395301
2018    25300.354074
2019    25832.442530
2020    25363.387058
2021    27410.174888
2022    27518.879625
2023    29481.997355
2024    31577.300853
2025    31843.562801
2026    32267.840113
Name: val_carga, dtype: float64

- Em média, a geração nos anos de 2026 e 2014 são maiores do que os seus anos anteriores

**(iii)**

- Nestes períodos, faz sentido pensar que a geração térmica tem um pico de utilização (para suprir as demandas do sistema)?

Para verificar se essa hipótese faz sentido, vamos verificar quais foram os meses com a maior média de uso, por tipo de geração, no ano de 2026

In [93]:
df_parquet_2026.groupby('mes')[['val_gerhidraulica',
                                'val_gereolica',
                                'val_gersolar',
                                'val_gertermica']].mean()

,val_gerhidraulica,val_gereolica,val_gersolar,val_gertermica
mes,,,,
1,20076.414900,4483.971735,4897.223996,3615.164041
2,23172.636373,3129.379671,4687.424113,3166.844248
3,22352.210749,3159.662987,4548.426685,3558.521270
4,20843.235987,4053.394600,4330.063545,2999.420586
5,18597.962709,4871.631082,3721.967486,3334.658499
6,17326.130099,5364.962702,3905.815145,4011.751131
7,16486.447132,5501.005200,4525.230677,4176.678233


- É fácil ver que a geração térmica teve um pico nos meses de junho e julho, ou seja, pode ser um indício de que a hipótese faz sentido.